## This notebook is to parecellate the TCP resting state fMRI data for cortical and subcortical regions

In [2]:

# IMPORTS
import numpy as np
import nibabel as nib # See here for install info if need be: https://nipy.org/nibabel/installation.html
# import hcp_utils as hcp # See here for install info if need be: https://pypi.org/project/hcp-utils/
from nibabel.cifti2 import BrainModelAxis
import pandas as pd

import pandas as pd
import os
from pathlib import Path
import glob




In [1]:
import nibabel as nib

def verify_cifti_alignment(dtseries_path, dlabel_path):
    dt = nib.load(dtseries_path)
    atl = nib.load(dlabel_path)
    
    # Get the BrainModelAxis (usually at index 1 for dtseries/dlabel)
    dt_axis = dt.header.get_axis(1)
    atl_axis = atl.header.get_axis(1)
    
    # 1. Check if the total number of grayordinates matches
    if len(dt_axis) != len(atl_axis):
        print(f"❌ LENGTH MISMATCH: Data ({len(dt_axis)}) vs Atlas ({len(atl_axis)})")
        return False

    # 2. Check individual structures
    # .iter_structures() returns (name, index_slice, brain_model)
    dt_structs = {name: bm for name, _, bm in dt_axis.iter_structures()}
    atl_structs = {name: bm for name, _, bm in atl_axis.iter_structures()}
    
    if dt_structs.keys() != atl_structs.keys():
        print(f"❌ STRUCTURE MISMATCH: Files contain different brain regions.")
        return False
        
    for name in dt_structs:
        dt_bm = dt_structs[name]
        atl_bm = atl_structs[name]
        
        # Check if the number of surface vertices or voxels matches for this structure
        if len(dt_bm.vertex) != len(atl_bm.vertex):
            print(f"❌ VERTEX MISMATCH in {name}: Data has {len(dt_bm.vertex)}, Atlas has {len(atl_bm.vertex)}")
            return False
            
    print("✅ Alignment Verified: Data and Atlas structures match perfectly.")
    return True

In [6]:
dtseries_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/participants_fmri/NDAR_INVAG023WG3/task-stroopAP_run-01_bold/task-stroopAP_run-01_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii"  # Change this path when needed
dlabel_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"  # Change this path
verify_cifti_alignment(dtseries_path, dlabel_path)

vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


✅ Alignment Verified: Data and Atlas structures match perfectly.


True

### Parcellate single file 

In [59]:
import nibabel as nb
import numpy as np
import pandas as pd

def parcellate_cifti(dtseries_path: str, atlas_dlabel_path: str, output_path: str = "parcell.txt") -> pd.DataFrame:
    """
    Parcellate a CIFTI dense timeseries (.dtseries.nii) using a .dlabel.nii atlas.
    Saves the result as a txt file.
    """
    # 1) Load dtseries and atlas
    dt = nb.load(dtseries_path)
    atl = nb.load(atlas_dlabel_path)
    
    # 2) Check axes match exactly
    dt_axis = dt.header.get_axis(1)
    atl_axis = atl.header.get_axis(1)
    assert dt_axis == atl_axis, (
        "Grayordinate axes differ! "
        "Run Workbench to align or resample your CIFTI files."
    )
    
    # 3) Extract labels vector, drop background (0)
    labels = np.squeeze(atl.get_fdata()).astype(int)
    roi_ids = np.unique(labels)
    roi_ids = roi_ids[roi_ids > 0]
    
    # 4) Precompute ROI indices
    roi_inds = {rid: np.flatnonzero(labels == rid) for rid in roi_ids}
    
    # 5) Load the full timeseries data into memory
    data = dt.get_fdata()  # This loads the data into memory
    assert data.ndim == 2 and data.shape[1] == labels.size, (
        f"Expected timeseries shape (T, {labels.size}), got {data.shape}."
    )
    
    # 6) Compute mean timecourse per ROI
    parcel_ts = np.column_stack([
        data[:, inds].mean(axis=1)
        for inds in roi_inds.values()
    ])
    
    # 7) Wrap in DataFrame
    df = pd.DataFrame(parcel_ts, columns=[f"ROI_{i}" for i in roi_ids])
    
    # 8) Save to txt file without column names or row indices
    df.to_csv(output_path, sep='\t', index=False, header=False, float_format='%.6f')
    print(f"Parcellated data saved to: {output_path}")
    print(f"Shape: {df.shape} (timepoints x ROIs)")
    
    return df


In [60]:
# Run the function
dt_path = "data/task-restAP_run-01_bold_Atlas_MSMAll_hp0_clean.dtseries.nii"
atlas_path = "data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"
df = parcellate_cifti(dt_path, atlas_path, "parcell.txt")

Parcellated data saved to: parcell.txt
Shape: (488, 232) (timepoints x ROIs)


## Batch processing

I've modified your code to handle multiple participants and their resting-state fMRI data. Here are the key changes:

**New Features:**
- **Batch processing**: Goes through all participant folders automatically
- **Rest folder detection**: Finds subfolders containing "rest" (case-insensitive)
- **Organized output**: Creates `parcellated_rsfmri` folder with participant subfolders
- **File naming**: Adds `parcellated_` prefix and changes extension to `.csv`
- **Error handling**: Continues processing even if individual files fail

**Output Structure:**
```
parcellated_rsfmri/
├── NDAR_INVAG900RVD/
│   ├── parcellated_file1.csv
│   └── parcellated_file2.csv
└── NDAR_PARTICIPANT2/
    └── parcellated_file3.csv
```

The script provides progress updates and error messages to help you track the processing. Simply run it in your Jupyter notebook and follow the prompts.

In [13]:

def parcellate_cifti(dtseries_path: str, atlas_dlabel_path: str, output_path: str = "parcell.txt"):
    """
    Parcellate a CIFTI dense timeseries (.dtseries.nii) using a .dlabel.nii atlas.
    Saves the result as a txt file.
    """
    # 1) Load dtseries and atlas
    dt = nib.load(dtseries_path)
    atl = nib.load(atlas_dlabel_path)
    
    # 2) Check axes match exactly
    dt_axis = dt.header.get_axis(1)
    atl_axis = atl.header.get_axis(1)
    assert dt_axis == atl_axis, (
        "Grayordinate axes differ! "
        "Run Workbench to align or resample your CIFTI files."
    )
    
    # 3) Extract labels vector, drop background (0)
    labels = np.squeeze(atl.get_fdata()).astype(int)
    roi_ids = np.unique(labels)
    roi_ids = roi_ids[roi_ids > 0]
    
    # 4) Precompute ROI indices
    roi_inds = {rid: np.flatnonzero(labels == rid) for rid in roi_ids}
    
    # 5) Load the full timeseries data into memory
    data = dt.get_fdata()  # This loads the data into memory
    assert data.ndim == 2 and data.shape[1] == labels.size, (
        f"Expected timeseries shape (T, {labels.size}), got {data.shape}."
    )
    
    # 6) Compute mean timecourse per ROI
    parcel_ts = np.column_stack([
        data[:, inds].mean(axis=1)
        for inds in roi_inds.values()
    ])
    
    # 7) Wrap in DataFrame
    df = pd.DataFrame(parcel_ts, columns=[f"ROI_{i}" for i in roi_ids])
    
    # 8) Save to txt file without column names or row indices
    df.to_csv(output_path, sep='\t', index=False, header=False, float_format='%.6f')
    print(f"Parcellated data saved to: {output_path}")
    print(f"Shape: {df.shape} (timepoints x ROIs)")

def process_all_participants(participant_data_path, parcellated_output_path, atlas_path):
    """
    Process all participants' resting-state fMRI data for parcellation.
    """
    
    # Expand paths (handles ~ for home directory)
    participant_data_path = os.path.expanduser(participant_data_path)
    parcellated_output_path = os.path.expanduser(parcellated_output_path)
    atlas_path = os.path.expanduser(atlas_path)
    
    # Validate input paths
    if not os.path.exists(participant_data_path):
        print(f"Error: Participant data path does not exist: {participant_data_path}")
        return
    
    if not os.path.exists(atlas_path):
        print(f"Error: Atlas file does not exist: {atlas_path}")
        return
    
    # Create main output directory
    parcellated_dir = Path(parcellated_output_path) / "parcellated_rsfmri"
    parcellated_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all participant folders
    participant_folders = [f for f in os.listdir(participant_data_path) 
                          if os.path.isdir(os.path.join(participant_data_path, f))]
    
    if not participant_folders:
        print(f"No participant folders found in: {participant_data_path}")
        return
    
    print(f"Found {len(participant_folders)} participant folders")
    
    processed_count = 0
    
    for participant_folder in participant_folders:
        participant_path = Path(participant_data_path) / participant_folder
        print(f"\nProcessing participant: {participant_folder}")
        
        # Create output folder for this participant
        participant_output_dir = parcellated_dir / participant_folder
        participant_output_dir.mkdir(exist_ok=True)
        
        # Find subfolders containing "rest"
        rest_folders = []
        for item in participant_path.iterdir():
            if item.is_dir() and "rest" in item.name.lower():
                rest_folders.append(item)
        
        if not rest_folders:
            print(f"  No 'rest' subfolders found for {participant_folder}")
            continue
        
        print(f"  Found {len(rest_folders)} rest folders: {[f.name for f in rest_folders]}")
        
        # Process each rest folder
        for rest_folder in rest_folders:
            print(f"    Processing folder: {rest_folder.name}")
            
            # Find CIFTI files (.dtseries.nii)
            cifti_files = list(rest_folder.glob("*.dtseries.nii"))
            
            if not cifti_files:
                print(f"      No .dtseries.nii files found in {rest_folder.name}")
                continue
            
            print(f"      Found {len(cifti_files)} CIFTI files")
            
            # Process each CIFTI file
            for cifti_file in cifti_files:
                try:
                    # Generate output filename
                    original_name = cifti_file.stem.replace('.dtseries', '')
                    output_filename = f"parcellated_{original_name}.csv"
                    output_file_path = participant_output_dir / output_filename
                    
                    print(f"        Parcellating: {cifti_file.name}")
                    
                    # Parcellate the CIFTI file
                    parcellate_cifti(
                        dtseries_path=str(cifti_file),
                        atlas_dlabel_path=atlas_path,
                        output_path=str(output_file_path)
                    )
                    
                    processed_count += 1
                    
                except Exception as e:
                    print(f"        Error processing {cifti_file.name}: {str(e)}")
                    continue
    
    print(f"\n=== Processing Complete ===")
    print(f"Successfully processed {processed_count} CIFTI files")
    print(f"Output directory: {parcellated_dir}")


In [14]:

# Set your paths here
participant_data_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/participants_fmri"  # Change this path when needed
parcellated_output_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data"  # Change this path  
atlas_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"  # Change this path

# Run the processing
process_all_participants(participant_data_path, parcellated_output_path, atlas_path)

vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value


Found 243 participant folders

Processing participant: NDAR_INVZL449UYG
  Found 4 rest folders: ['task-restPA_run-01_bold', 'task-restAP_run-01_bold', 'task-restAP_run-02_bold', 'task-restPA_run-02_bold']
    Processing folder: task-restPA_run-01_bold
      Found 1 CIFTI files
        Parcellating: task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii


vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri/NDAR_INVZL449UYG/parcellated_task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (488, 232) (timepoints x ROIs)
    Processing folder: task-restAP_run-01_bold
      Found 1 CIFTI files
        Parcellating: task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii


vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value


...

Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri/NDAR_INVRF868XAA/parcellated_task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (488, 232) (timepoints x ROIs)
    Processing folder: task-restAP_run-01_bold
      Found 1 CIFTI files
        Parcellating: task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii


vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri/NDAR_INVRF868XAA/parcellated_task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (488, 232) (timepoints x ROIs)
    Processing folder: task-restAP_run-02_bold
      Found 1 CIFTI files
        Parcellating: task-restAP_run-02_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii


vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631480) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri/NDAR_INVRF868XAA/parcellated_task-restAP_run-02_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (488, 232) (timepoints x ROIs)
    Processing folder: task-restPA_run-02_bold
      Found 1 CIFTI files
        Parcellating: task-restPA_run-02_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii
Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri/NDAR_INVRF868XAA/parcellated_task-restPA_run-02_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (488, 232) (timepoints x ROIs)

=== Processing Complete ===
Successfully processed 954 CIFTI files
Output directory: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri


### Concatenating run 01, 02, AP and PA
**Excluding participants that miss runs:** 
- NDAR_INVCM621YRY 
- NDAR_INVFE128JJV
- NDAR_INVGB371PPV
- NDAR_INVJT253NWQ
- NDAR_INVJV338PGX
- NDAR_INVTF281GWR



In [16]:
import pandas as pd
import os
from pathlib import Path


def concatenate_participant_csvs(parcellated_data_path):
    """
    Concatenate the 4 CSV files for each participant in the specified order.
    Provides detailed reporting about which participants were skipped and why.
    """
    # Expand path (handles ~ for home directory)
    parcellated_data_path = os.path.expanduser(parcellated_data_path)

    # Define the order for concatenation
    file_order = [
        "parcellated_task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv",
        "parcellated_task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv",
        "parcellated_task-restAP_run-02_bold_Atlas_MSMAll_hp2000_clean.csv",
        "parcellated_task-restPA_run-02_bold_Atlas_MSMAll_hp2000_clean.csv"
    ]

    # Check if directory exists
    if not os.path.exists(parcellated_data_path):
        print(f"Error: Directory does not exist: {parcellated_data_path}")
        return

    # Get all participant folders
    participant_folders = [
        f for f in os.listdir(parcellated_data_path)
        if os.path.isdir(os.path.join(parcellated_data_path, f))
    ]

    if not participant_folders:
        print(f"No participant folders found in: {parcellated_data_path}")
        return

    print(f"Found {len(participant_folders)} participant folders")

    successful_concatenations = 0
    already_exists = 0
    skipped_participants = []
    error_participants = []

    for participant_folder in participant_folders:
        participant_path = Path(parcellated_data_path) / participant_folder
        print(f"\nProcessing participant: {participant_folder}")

        # Check if concatenated file already exists
        output_filename = (
            f"{participant_folder}_concatenated_parcellated_task-rest_bold_Atlas_MSMAll_hp2000_clean.csv"
        )
        output_path = participant_path / output_filename

        if output_path.exists():
            print(f"  ✓ Concatenated file already exists - skipping")
            already_exists += 1
            continue

        # Check if all 4 CSV files exist
        missing_files = []
        existing_files = []

        for filename in file_order:
            file_path = participant_path / filename
            if file_path.exists():
                existing_files.append(file_path)
            else:
                missing_files.append(filename)

        if missing_files:
            print(f"  ❌ Missing files: {[os.path.basename(f) for f in missing_files]}")
            print(f"  Found {len(existing_files)} out of 4 files - skipping participant")
            skipped_participants.append({
                'participant': participant_folder,
                'reason': 'missing_files',
                'missing_files': [os.path.basename(f) for f in missing_files],
                'available_files': len(existing_files)
            })
            continue

        # Load and concatenate CSV files
        try:
            dataframes = []
            total_rows = 0

            for i, file_path in enumerate(existing_files):
                print(f"  Loading: {file_path.name}")
                df = pd.read_csv(file_path, header=None, sep='\t')
                dataframes.append(df)
                total_rows += len(df)
                print(f"    Shape: {df.shape}")

            # Concatenate all dataframes
            concatenated_df = pd.concat(dataframes, axis=0, ignore_index=True)
            print(f"  Concatenated shape: {concatenated_df.shape}")

            # Save concatenated file
            concatenated_df.to_csv(
                output_path,
                sep='\t',
                index=False,
                header=False,
                float_format='%.6f'
            )
            print(f"  ✓ Saved: {output_filename}")

            successful_concatenations += 1

        except Exception as e:
            print(f"  ❌ Error processing {participant_folder}: {str(e)}")
            error_participants.append({
                'participant': participant_folder,
                'reason': 'processing_error',
                'error': str(e)
            })
            continue

    # Detailed summary
    print(f"\n{'='*80}")
    print(f"DETAILED CONCATENATION SUMMARY")
    print(f"{'='*80}")
    print(f"Total participant folders found: {len(participant_folders)}")
    print(f"Already had concatenated files: {already_exists}")
    print(f"Successfully concatenated: {successful_concatenations}")
    print(f"Skipped due to missing files: {len(skipped_participants)}")
    print(f"Failed due to errors: {len(error_participants)}")
    print(f"Total with concatenated files: {already_exists + successful_concatenations}")

    # Report skipped participants
    if skipped_participants:
        print(f"\n{'='*80}")
        print(f"PARTICIPANTS SKIPPED DUE TO MISSING FILES ({len(skipped_participants)}):")
        print(f"{'='*80}")
        for skip_info in skipped_participants:
            print(f"• {skip_info['participant']}")
            print(f"  Available files: {skip_info['available_files']}/4")
            print(f"  Missing files: {skip_info['missing_files']}")

    # Report error participants
    if error_participants:
        print(f"\n{'='*80}")
        print(f"PARTICIPANTS WITH PROCESSING ERRORS ({len(error_participants)}):")
        print(f"{'='*80}")
        for error_info in error_participants:
            print(f"• {error_info['participant']}")
            print(f"  Error: {error_info['error']}")

    print(f"\n{'='*80}")


In [18]:
# Set your path here
parcellated_data_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_rsfmri"

# Run the concatenation
concatenate_participant_csvs(parcellated_data_path)

Found 237 participant folders

Processing participant: NDAR_INVZL449UYG
  Loading: parcellated_task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restAP_run-02_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restPA_run-02_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Concatenated shape: (1952, 232)
  ✓ Saved: NDAR_INVZL449UYG_concatenated_parcellated_task-rest_bold_Atlas_MSMAll_hp2000_clean.csv

Processing participant: NDAR_INVVV366BKJ
  Loading: parcellated_task-restAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restAP_run-02_bold_Atlas_MSMAll_hp2000_clean.csv
    Shape: (488, 232)
  Loading: parcellated_task-restPA_run-02_bold_Atlas_MSMAll_h

### Parcellating Hammer (Emotional Face) task fMRI data

In [15]:

def parcellate_cifti(dtseries_path: str, atlas_dlabel_path: str, output_path: str = "parcell.txt"):
    """
    Parcellate a CIFTI dense timeseries (.dtseries.nii) using a .dlabel.nii atlas.
    Saves the result as a txt file.
    """
    # 1) Load dtseries and atlas
    dt = nib.load(dtseries_path)
    atl = nib.load(atlas_dlabel_path)
    
    # 2) Check axes match exactly
    dt_axis = dt.header.get_axis(1)
    atl_axis = atl.header.get_axis(1)
    assert dt_axis == atl_axis, (
        "Grayordinate axes differ! "
        "Run Workbench to align or resample your CIFTI files."
    )
    
    # 3) Extract labels vector, drop background (0)
    labels = np.squeeze(atl.get_fdata()).astype(int)
    roi_ids = np.unique(labels)
    roi_ids = roi_ids[roi_ids > 0]
    
    # 4) Precompute ROI indices
    roi_inds = {rid: np.flatnonzero(labels == rid) for rid in roi_ids}
    
    # 5) Load the full timeseries data into memory
    data = dt.get_fdata()  # This loads the data into memory
    assert data.ndim == 2 and data.shape[1] == labels.size, (
        f"Expected timeseries shape (T, {labels.size}), got {data.shape}."
    )
    
    # 6) Compute mean timecourse per ROI
    parcel_ts = np.column_stack([
        data[:, inds].mean(axis=1)
        for inds in roi_inds.values()
    ])
    
    # 7) Wrap in DataFrame
    df = pd.DataFrame(parcel_ts, columns=[f"ROI_{i}" for i in roi_ids])
    
    # 8) Save to txt file without column names or row indices
    df.to_csv(output_path, sep='\t', index=False, header=False, float_format='%.6f')
    print(f"Parcellated data saved to: {output_path}")
    print(f"Shape: {df.shape} (timepoints x ROIs)")

def process_all_participants(participant_data_path, parcellated_output_path, atlas_path):
    """
    Process all participants' resting-state fMRI data for parcellation.
    """
    
    # Expand paths (handles ~ for home directory)
    participant_data_path = os.path.expanduser(participant_data_path)
    parcellated_output_path = os.path.expanduser(parcellated_output_path)
    atlas_path = os.path.expanduser(atlas_path)
    
    # Validate input paths
    if not os.path.exists(participant_data_path):
        print(f"Error: Participant data path does not exist: {participant_data_path}")
        return
    
    if not os.path.exists(atlas_path):
        print(f"Error: Atlas file does not exist: {atlas_path}")
        return
    
    # Create main output directory
    parcellated_dir = Path(parcellated_output_path) / "parcellated_sfmri"
    parcellated_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all participant folders
    participant_folders = [f for f in os.listdir(participant_data_path) 
                          if os.path.isdir(os.path.join(participant_data_path, f))]
    
    if not participant_folders:
        print(f"No participant folders found in: {participant_data_path}")
        return
    
    print(f"Found {len(participant_folders)} participant folders")
    
    processed_count = 0
    
    for participant_folder in participant_folders:
        participant_path = Path(participant_data_path) / participant_folder
        print(f"\nProcessing participant: {participant_folder}")
        
        # Create output folder for this participant
        participant_output_dir = parcellated_dir / participant_folder
        participant_output_dir.mkdir(exist_ok=True)
        
        # 1. Update folder filter: Find folders containing "hammer"
        hammer_folders = []
        for item in participant_path.iterdir():
            if item.is_dir() and "hammer" in item.name.lower():
                hammer_folders.append(item)
        
        if not hammer_folders:
            print(f"  No 'hammer' subfolders found for {participant_folder}")
            continue
        
        # 2. Process the identified folders
        for hammer_folder in hammer_folders:
            # 3. Update file filter: Target the specific filename pattern
            # Using a specific string ensures we don't grab unwanted files
            target_pattern = "*hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.dtseries.nii"
            cifti_files = list(hammer_folder.glob(target_pattern))
            
            if not cifti_files:
                print(f"      Specific hammer file not found in {hammer_folder.name}")
                continue
            
            # Process the specific file(s) found
            for cifti_file in cifti_files:
                try:
                    # [Keep the rest of the original processing/parcellation logic here]
                    original_name = cifti_file.stem.replace('.dtseries', '')
                    output_filename = f"parcellated_{original_name}.csv"
                    output_file_path = participant_output_dir / output_filename
                    
                    parcellate_cifti(
                        dtseries_path=str(cifti_file),
                        atlas_dlabel_path=atlas_path,
                        output_path=str(output_file_path)
                    )
                    processed_count += 1
                except Exception as e:
                    print(f"        Error: {str(e)}")
    
    print(f"\n=== Processing Complete ===")
    print(f"Successfully processed {processed_count} CIFTI files")
    print(f"Output directory: {parcellated_dir}")


In [12]:

# Set your paths here
participant_data_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/participants_fmri"  # Change this path when needed
parcellated_output_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data"  # Change this path  
atlas_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"  # Change this path

# Run the processing
process_all_participants(participant_data_path, parcellated_output_path, atlas_path)

vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
/Users/proghani/.pyenv/versions/3.10.7/lib/python3.10/site-packages/nibabel/nifti1.py:617: UserWarning: Extension size is not a multiple of 16 bytes; Assuming size is correct and hoping for the best
  warnings.warn(
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Found 243 participant folders

Processing participant: NDAR_INVZL449UYG


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri/NDAR_INVZL449UYG/parcellated_task-hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (493, 232) (timepoints x ROIs)

Processing participant: NDAR_INVVV366BKJ


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


...

Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri/NDAR_INVFU389BX1/parcellated_task-hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (493, 232) (timepoints x ROIs)

Processing participant: NDAR_INVPG851GUA


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri/NDAR_INVPG851GUA/parcellated_task-hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (493, 232) (timepoints x ROIs)

Processing participant: NDAR_INVAM061NXD


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri/NDAR_INVAM061NXD/parcellated_task-hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (493, 232) (timepoints x ROIs)

Processing participant: NDAR_INVRF868XAA
Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri/NDAR_INVRF868XAA/parcellated_task-hammerAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (493, 232) (timepoints x ROIs)

=== Processing Complete ===
Successfully processed 226 CIFTI files
Output directory: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_sfmri


### Parcellating Hammer (Emotional Face) task fMRI data

In [19]:
def parcellate_cifti(dtseries_path: str, atlas_dlabel_path: str, output_path: str = "parcell.txt"):
    """
    Parcellate a CIFTI dense timeseries (.dtseries.nii) using a .dlabel.nii atlas.
    Saves the result as a txt file.
    """
    # 1) Load dtseries and atlas
    dt = nib.load(dtseries_path)
    atl = nib.load(atlas_dlabel_path)
    
    # 2) Check axes match exactly
    dt_axis = dt.header.get_axis(1)
    atl_axis = atl.header.get_axis(1)
    assert dt_axis == atl_axis, (
        "Grayordinate axes differ! "
        "Run Workbench to align or resample your CIFTI files."
    )
    
    # 3) Extract labels vector, drop background (0)
    labels = np.squeeze(atl.get_fdata()).astype(int)
    roi_ids = np.unique(labels)
    roi_ids = roi_ids[roi_ids > 0]
    
    # 4) Precompute ROI indices
    roi_inds = {rid: np.flatnonzero(labels == rid) for rid in roi_ids}
    
    # 5) Load the full timeseries data into memory
    data = dt.get_fdata()  # This loads the data into memory
    assert data.ndim == 2 and data.shape[1] == labels.size, (
        f"Expected timeseries shape (T, {labels.size}), got {data.shape}."
    )
    
    # 6) Compute mean timecourse per ROI
    parcel_ts = np.column_stack([
        data[:, inds].mean(axis=1)
        for inds in roi_inds.values()
    ])
    
    # 7) Wrap in DataFrame
    df = pd.DataFrame(parcel_ts, columns=[f"ROI_{i}" for i in roi_ids])
    
    # 8) Save to txt file without column names or row indices
    df.to_csv(output_path, sep='\t', index=False, header=False, float_format='%.6f')
    print(f"Parcellated data saved to: {output_path}")
    print(f"Shape: {df.shape} (timepoints x ROIs)")

def process_all_participants(participant_data_path, parcellated_output_path, atlas_path):
    """
    Process all participants' resting-state fMRI data for parcellation.
    """
    
    # Expand paths (handles ~ for home directory)
    participant_data_path = os.path.expanduser(participant_data_path)
    parcellated_output_path = os.path.expanduser(parcellated_output_path)
    atlas_path = os.path.expanduser(atlas_path)
    
    # Validate input paths
    if not os.path.exists(participant_data_path):
        print(f"Error: Participant data path does not exist: {participant_data_path}")
        return
    
    if not os.path.exists(atlas_path):
        print(f"Error: Atlas file does not exist: {atlas_path}")
        return
    
    # Create main output directory
    parcellated_dir = Path(parcellated_output_path) / "parcellated_stroop_task_fmri"
    parcellated_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all participant folders
    participant_folders = [f for f in os.listdir(participant_data_path) 
                          if os.path.isdir(os.path.join(participant_data_path, f))]
    
    if not participant_folders:
        print(f"No participant folders found in: {participant_data_path}")
        return
    
    print(f"Found {len(participant_folders)} participant folders")
    
    processed_count = 0
    
    for participant_folder in participant_folders:
        participant_path = Path(participant_data_path) / participant_folder
        print(f"\nProcessing participant: {participant_folder}")
        
        # Create output folder for this participant
        participant_output_dir = parcellated_dir / participant_folder
        participant_output_dir.mkdir(exist_ok=True)
        
        # 1. Update folder filter: Find folders containing "stroop"
        stroop_folders = []
        for item in participant_path.iterdir():
            if item.is_dir() and "stroop" in item.name.lower():
                stroop_folders.append(item)
        
        if not stroop_folders:
            print(f"  No 'stroop' subfolders found for {participant_folder}")
            continue
        
        print(f"  Found {len(stroop_folders)} stroop folders: {[f.name for f in stroop_folders]}")
        
        # 2. Process each stroop folder
        for stroop_folder in stroop_folders:
            # 3. Update file filter: Find CIFTI files starting with "task-stroop"
            # The '*' at the end acts as a wildcard for the rest of the filename
            cifti_files = list(stroop_folder.glob("task-stroop*.dtseries.nii"))
            
            if not cifti_files:
                print(f"      No 'task-stroop' .dtseries.nii files found in {stroop_folder.name}")
                continue
            
            # Process the files
            for cifti_file in cifti_files:
                try:
                    original_name = cifti_file.stem.replace('.dtseries', '')
                    output_filename = f"parcellated_{original_name}.csv"
                    output_file_path = participant_output_dir / output_filename
                    
                    parcellate_cifti(
                        dtseries_path=str(cifti_file),
                        atlas_dlabel_path=atlas_path,
                        output_path=str(output_file_path)
                    )
                    processed_count += 1
                except Exception as e:
                    print(f"        Error processing {cifti_file.name}: {str(e)}")
    
    print(f"\n=== Processing Complete ===")
    print(f"Successfully processed {processed_count} CIFTI files")
    print(f"Output directory: {parcellated_dir}")


In [20]:

# Set your paths here
participant_data_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/participants_fmri"  # Change this path when needed
parcellated_output_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data"  # Change this path  
atlas_path = "~/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"  # Change this path

# Run the processing
process_all_participants(participant_data_path, parcellated_output_path, atlas_path)

vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
/Users/proghani/.pyenv/versions/3.10.7/lib/python3.10/site-packages/nibabel/nifti1.py:617: UserWarning: Extension size is not a multiple of 16 bytes; Assuming size is correct and hoping for the best
  warnings.warn(
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Found 243 participant folders

Processing participant: NDAR_INVZL449UYG
  Found 2 stroop folders: ['task-stroopPA_run-01_bold', 'task-stroopAP_run-01_bold']


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri/NDAR_INVZL449UYG/parcellated_task-stroopPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (510, 232) (timepoints x ROIs)


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


...

Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri/NDAR_INVAM061NXD/parcellated_task-stroopPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (510, 232) (timepoints x ROIs)


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri/NDAR_INVAM061NXD/parcellated_task-stroopAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (510, 232) (timepoints x ROIs)

Processing participant: NDAR_INVRF868XAA
  Found 2 stroop folders: ['task-stroopPA_run-01_bold', 'task-stroopAP_run-01_bold']


vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value
vox offset (=631496) not divisible by 16, not SPM compatible; leaving at current value


Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri/NDAR_INVRF868XAA/parcellated_task-stroopPA_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (510, 232) (timepoints x ROIs)
Parcellated data saved to: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri/NDAR_INVRF868XAA/parcellated_task-stroopAP_run-01_bold_Atlas_MSMAll_hp2000_clean.csv
Shape: (510, 232) (timepoints x ROIs)

=== Processing Complete ===
Successfully processed 450 CIFTI files
Output directory: /Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/parcellated_stroop_task_fmri
